# Chapter 9.6 - Concise Implementation of Recurrent Neural Networks

PyTorch's `nn.RNN` packages the recurrent loop into a tested layer. This notebook maps its exact tensor contracts to a character-level language model, trains it on a tiny offline pattern, and decodes predictions.

## How to use this notebook

Run the notebook from top to bottom in a clean kernel. Everything is generated from small tensors or inline text, so there are no downloads. Before important cells, predict the time, batch, feature, vocabulary, and hidden-state shapes. Treat every assertion as an executable contract rather than decoration.

## You are done when you can

- read the input, output, and hidden-state contracts of nn.RNN
- explain the roles of an embedding, recurrent layer, and output projection
- handle `batch_first=True` without transposing axes accidentally
- train and decode with a concise RNN language model
- diagnose hidden-state errors involving layers and batch size


In [ ]:
import math
import random
from collections import Counter

import torch
from torch import nn
from torch.nn import functional as F

torch.manual_seed(0)
random.seed(0)
torch.set_printoptions(precision=4, sci_mode=False)

def shape(x):
    return tuple(x.shape)


## 9.6.0 The Problem This Notebook Solves

The scratch model made every equation visible. A framework layer reduces repeated code and offers optimized kernels, but it introduces a strict interface.

With `batch_first=True`, an `nn.RNN` receives `(batch, time, input_size)` and returns:

```text
outputs:    (batch, time, hidden_size)
final_state: (num_layers, batch, hidden_size)
```

The leading state axis exists because a stacked RNN stores one final state per recurrent layer. It remains present even when `num_layers=1`.


## 9.6.1 Defining the Model

An **embedding** is a trainable lookup table shaped `(vocab_size, embedding_size)`. It converts integer token IDs to dense feature vectors. An RNN then processes those vectors through time. A linear projection turns every hidden vector into `vocab_size` next-token logits.

`super().__init__()` initializes the `nn.Module` part of the object before child modules are assigned. Without it, PyTorch cannot reliably register the embedding, RNN, and projection.


In [ ]:
class ConciseRNNLM(nn.Module):
    def __init__(self, vocab_size, embedding_size, hidden_size, num_layers=1):
        super().__init__()
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.embedding = nn.Embedding(vocab_size, embedding_size)
        self.rnn = nn.RNN(
            input_size=embedding_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
        )
        self.output = nn.Linear(hidden_size, vocab_size)

    def begin_state(self, batch_size, device=None):
        return torch.zeros(self.num_layers, batch_size, self.hidden_size, device=device)

    def forward(self, token_ids, state=None):
        embedded = self.embedding(token_ids)
        recurrent_outputs, final_state = self.rnn(embedded, state)
        logits = self.output(recurrent_outputs)
        return logits, final_state


In [ ]:
model = ConciseRNNLM(vocab_size=7, embedding_size=5, hidden_size=9, num_layers=2)
X = torch.tensor([[0, 1, 2, 3], [3, 2, 1, 0]])
state = model.begin_state(batch_size=X.shape[0], device=X.device)
embedded = model.embedding(X)
logits, final_state = model(X, state)

print("IDs:", shape(X))
print("embedded:", shape(embedded))
print("logits:", shape(logits))
print("final state:", shape(final_state))
assert shape(embedded) == (2, 4, 5)
assert shape(logits) == (2, 4, 7)
assert shape(final_state) == (2, 2, 9)


The scratch notebook flattened logits in time-major order. This model preserves batch-first axes, so `logits.reshape(-1, vocab)` aligns directly with `Y.reshape(-1)`. Neither convention is inherently better. Mixing conventions is the bug.


## 9.6.2 Training and Predicting

We train on the same repeating seven-token pattern. Each training call starts from zeros, so it does not carry state between epochs. The single full sequence is small enough that backpropagation through all its time steps is safe for this drill.


In [ ]:
corpus = torch.tensor(([0, 1, 2, 3, 4, 5, 6] * 12), dtype=torch.long)
X_train = corpus[:-1].reshape(1, -1)
Y_train = corpus[1:].reshape(1, -1)
model = ConciseRNNLM(vocab_size=7, embedding_size=8, hidden_size=16)
optimizer = torch.optim.Adam(model.parameters(), lr=0.03)

losses = []
for epoch in range(60):
    optimizer.zero_grad()
    logits, _ = model(X_train)
    loss = F.cross_entropy(logits.reshape(-1, 7), Y_train.reshape(-1))
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    losses.append(loss.item())

print("first and last loss:", losses[0], losses[-1])
assert math.isfinite(losses[-1])
assert losses[-1] < losses[0]


In [ ]:
def generate(prefix, length, model):
    model.eval()
    state = None
    output = list(prefix)
    with torch.no_grad():
        for token in prefix[:-1]:
            _, state = model(torch.tensor([[token]]), state)
        for _ in range(length):
            logits, state = model(torch.tensor([[output[-1]]]), state)
            output.append(int(logits[0, -1].argmax()))
    return output

generated = generate([0, 1], length=12, model=model)
print(generated)
assert generated[:2] == [0, 1]
assert len(generated) == 14


## 9.6.3 Inspect What the Concise Layer Owns

For each RNN layer, PyTorch registers an input-to-hidden weight, hidden-to-hidden weight, and usually two biases. Layer 0 receives embedding features. Higher layers receive hidden outputs from the layer below.


In [ ]:
two_layer = ConciseRNNLM(vocab_size=7, embedding_size=5, hidden_size=9, num_layers=2)
for name, parameter in two_layer.rnn.named_parameters():
    print(name, shape(parameter))

names = dict(two_layer.rnn.named_parameters())
assert shape(names["weight_ih_l0"]) == (9, 5)
assert shape(names["weight_hh_l0"]) == (9, 9)
assert shape(names["weight_ih_l1"]) == (9, 9)


## 9.6.4 Break It Deliberately: Missing the Layer Axis

For a two-layer RNN and batch size two, state must be `(2, 2, hidden_size)`. A tensor shaped only `(batch, hidden_size)` looks like the scratch state but violates the concise layer contract.


In [ ]:
wrong_state = torch.zeros(2, two_layer.hidden_size)
try:
    two_layer(torch.tensor([[0, 1], [2, 3]]), wrong_state)
except RuntimeError as error:
    print(type(error).__name__)
    print(str(error).splitlines()[0])
else:
    raise AssertionError("Expected a missing recurrent-layer axis to fail")


## 9.6 Checkpoint

Answer these without rerunning the notebook. Short markdown answers are enough.

1. What does an embedding do that a raw integer token ID does not?
2. Why does final RNN state have a num_layers axis?
3. When can batch-first labels be flattened without transposing?
4. What work does nn.RNN replace from the scratch implementation?
5. Why is a framework layer still not shape magic?
